# USAAIO AI Olympiads 1 - Session 1 Homework

This single Colab notebook combines the Session 1 coding notebook, the required LaTeX practice, and typed solutions for the Day 1 extra problems.

# Week 1 --- Live Coding: Reading an Olympiad Problem

**Alpha Star Academy --- AI Olympiads 1**

Olympiad problems rarely hand you a model to train. They hand you a *machine you can only poke* and ask what it is. Today we take an actual USAAIO Round 1 problem --- recover a matrix from how it acts on vectors, then split it into the smallest possible sum of outer products --- and we solve it three ways: on paper, in code, and as a grader would read it.

Then two conceptual problems that decide contest points: **which tasks are unsupervised** (and the trap that hides inside clustering), and **what actually differs between L1 and L2 at the same lambda**.

Every cell uses **tiny data**, so it runs instantly with no downloads. Run the setup cell first, then type each snippet in order.

**Three hours, three chunks:** the matrix problem (Sections 1-4) -> the two concept problems (Sections 5-6) -> contest discipline and a full solve (Sections 7-8).

In [ ]:
#@title Setup -- run this first
import numpy as np
np.set_printoptions(suppress=True)

def A_action(x):
    """The mystery machine. Feed it a length-4 vector, get a length-3 vector back.
    In the real problem you are told ONLY this rule -- never the matrix itself."""
    x0, x1, x2, x3 = x
    return np.array([x2 - x3, x3, x0 + 2*x1], dtype=float)

x = np.array([1., 2., 3., 4.])     # a test vector we reuse all notebook
print("setup ready")

---
# Section 1 --- A matrix is a function

*The problem says: there is a matrix $A \in \mathbb{R}^{3\times 4}$, and for any $x$, $Ax = (x_2-x_3,\ x_3,\ x_0+2x_1)^\top$. Find $A$.*

In [ ]:
y = A_action(x)
print("input :", x)
print("output:", y)
print("shapes:", x.shape, "->", y.shape)

**Expected:** `input : [1. 2. 3. 4.]`, `output: [-1.  4.  5.]`, `shapes: (4,) -> (3,)`. The machine eats a vector of length 4 and returns one of length 3 --- so $A$ must be $3 \times 4$. The *output* length is the row count, the *input* length is the column count.

In [ ]:
p = np.array([1., 2., 3., 4.])
q = np.array([0., 3., 7., -3.])
left = A_action(p + q)
right = A_action(p) + A_action(q)
print(left)
print(right)
print("linear?", np.allclose(left, right))

**Expected:** both `[ 9.  1. 11.]`, `linear? True`. This is the permission slip for everything that follows. A matrix is exactly a function that respects scaling and addition --- so if we learn what it does to a few well-chosen inputs, we know what it does to *every* input.

In [ ]:
zero = np.zeros(4)
print(A_action(zero))

**Expected:** `[0. 0. 0.]`. Every linear map sends zero to zero. If a rule you are handed does not, no matrix can represent it --- check this first, it takes one line and occasionally saves a whole problem.

---
# Section 2 --- Recovering A by probing

In [ ]:
basis4 = np.eye(4)
columns = [A_action(e_j) for e_j in basis4]
for column in columns:
    print(column)

**Expected:** `[0. 0. 1.]`, `[0. 0. 2.]`, `[1. 0. 0.]`, `[-1.  1.  0.]`. Feeding in the $j$-th standard basis vector returns the $j$-th **column** of $A$ --- because $Ae_j$ picks out exactly that column. Four probes, four columns, done.

In [ ]:
A = np.column_stack(columns)
print(A)
print("shape:", A.shape)

**Expected:**
```
[[ 0.  0.  1. -1.]
 [ 0.  0.  0.  1.]
 [ 1.  2.  0.  0.]]
```
`shape: (3, 4)`. Read row 0 as the recipe for output 0: $0\cdot x_0 + 0\cdot x_1 + 1\cdot x_2 - 1\cdot x_3 = x_2 - x_3$. Every row of $A$ is one output coordinate's recipe.

In [ ]:
print(A @ x)
print(A_action(x))
print("match?", np.allclose(A @ x, A_action(x)))

**Expected:** both `[-1.  4.  5.]`, `match? True`. Never submit a recovered matrix without this check --- it costs one line and catches every sign and transpose error.

In [ ]:
rng = np.random.default_rng(7)
tests = rng.normal(size=(6, 4))
print(all(np.allclose(A @ z, A_action(z)) for z in tests))

**Expected:** `True`. One test vector can pass by luck; six random ones cannot. Cheap randomised verification like this is the closest thing you get to a grader before the grader.

---
# Section 3 --- Outer products: the rank-1 atom

*Part 2 of the problem asks for $A=\sum_{i=0}^{I-1} u^{(i)}v^{(i)\top}$ with $I$ as small as possible. First: what is one such term?*

In [ ]:
u0 = np.array([1., 0., 0.])
v0 = np.array([0., 0., 1., -1.])
term0 = np.outer(u0, v0)
print(term0)
print("rank:", np.linalg.matrix_rank(term0))

**Expected:** a $3\times 4$ matrix whose only nonzero row is `[ 0.  0.  1. -1.]`, and `rank: 1`. A column vector times a row vector is a full matrix --- and it always has rank exactly 1. That is the atom we are building from.

In [ ]:
measurement = v0 @ x
spread = u0 * measurement
print(measurement)
print(spread)
print(term0 @ x)

**Expected:** `-1.0`, then `[-1.  0.  0.]` twice (signs may print as `-0.`). This is the sentence to memorise: **an outer product measures the input in the direction $v$, then spreads that single number along $u$.** Attention scores, PCA components, and low-rank adapters are all this same move.

In [ ]:
u1 = np.array([0., 1., 0.])
v1 = np.array([0., 0., 0., 1.])
two_terms = term0 + np.outer(u1, v1)
print(np.linalg.matrix_rank(two_terms))

**Expected:** `2`. Each term can add at most 1 to the rank. So a sum of $I$ outer products has rank $\le I$ --- which means $I \ge \text{rank}(A)$, and that inequality is what makes the next section's answer forced rather than guessed.

---
# Section 4 --- The minimal decomposition

In [ ]:
u2 = np.array([0., 0., 1.])
v2 = np.array([1., 2., 0., 0.])
A_decomposed = term0 + np.outer(u1, v1) + np.outer(u2, v2)
print(A_decomposed)
print("matches A?", np.allclose(A_decomposed, A))

**Expected:** the same matrix as Section 2, `matches A? True`. The construction is not clever: take $u^{(i)} = e_i$ (the $i$-th output slot) and $v^{(i)}$ = the $i$-th row of $A$. **Any** matrix decomposes this way, with $I$ = number of nonzero rows.

In [ ]:
rank_A = np.linalg.matrix_rank(A)
singular_values_A = np.linalg.svd(A, compute_uv=False)
print("rank(A) =", rank_A)
print(np.round(singular_values_A, 3))

**Expected:** `rank(A) = 3` and `[2.236 1.618 0.618]`. Three nonzero singular values means rank 3. Combined with $I \ge \text{rank}(A)$ from Section 3, our three-term answer is **minimal**: $I = 3$. That argument, written out, is what earns the points --- the decomposition alone does not.

In [ ]:
U_A, s_A, Vt_A = np.linalg.svd(A, full_matrices=False)
A_rank2 = (U_A[:, :2] * s_A[:2]) @ Vt_A[:2, :]
print(round(np.linalg.norm(A - A_rank2, 2), 3))
print(round(np.linalg.norm(A - A, 2), 3))
print(round(np.linalg.norm(A - A_rank2, "fro"), 3))

**Expected:** `0.618`, `0.0`, `0.618`. Two terms cannot reach $A$ --- the best they achieve is off by exactly the smallest singular value. This is proof by construction that $I=2$ is impossible, and it previews PCA (Week 9), which is this same truncation used deliberately.

# **Part 3.2**

We need the smallest integer $I$ for which

$$
A=\sum_{i=0}^{I-1}u^{(i)}v^{(i)\top},
\qquad
A=
\begin{pmatrix}
0&0&1&-1\\
0&0&0&1\\
1&2&0&0
\end{pmatrix}.
$$

**Achievability.** Three terms are enough. Choose

$$
u^{(0)}=\begin{pmatrix}1\\0\\0\end{pmatrix},\quad
v^{(0)}=\begin{pmatrix}0\\0\\1\\-1\end{pmatrix},\qquad
u^{(1)}=\begin{pmatrix}0\\1\\0\end{pmatrix},\quad
v^{(1)}=\begin{pmatrix}0\\0\\0\\1\end{pmatrix},
$$

$$
u^{(2)}=\begin{pmatrix}0\\0\\1\end{pmatrix},\qquad
v^{(2)}=\begin{pmatrix}1\\2\\0\\0\end{pmatrix}.
$$

Then each $u^{(i)}v^{(i)\top}$ places the row $v^{(i)\top}$ into the corresponding row of the matrix, so

$$
A=u^{(0)}v^{(0)\top}+u^{(1)}v^{(1)\top}+u^{(2)}v^{(2)\top}.
$$

Therefore $I\leq 3$.

**Minimality.** Every outer product $u^{(i)}v^{(i)\top}$ has rank at most $1$. By subadditivity of rank,

$$
\operatorname{rank}\!\left(\sum_{i=0}^{I-1}u^{(i)}v^{(i)\top}\right)\leq I.
$$

The three rows of $A$ are linearly independent. In a relation among the first two rows, the third coordinate first forces the coefficient of row $1$ to be zero, and the fourth coordinate then forces the coefficient of row $2$ to be zero. Row $3$ is independent of those two because it has support in the first two columns while they have support in the last two columns. Hence $\operatorname{rank}(A)=3$, so any valid decomposition must satisfy $I\geq 3$.

Combining $I\leq 3$ and $I\geq 3$, the smallest possible value is

$$
\boxed{I=3}.
$$

In [ ]:
# Verify both halves of the Part 3.2 argument computationally.
part_32_terms = [
    np.outer(np.array([1., 0., 0.]), np.array([0., 0., 1., -1.])),
    np.outer(np.array([0., 1., 0.]), np.array([0., 0., 0., 1.])),
    np.outer(np.array([0., 0., 1.]), np.array([1., 2., 0., 0.])),
]
part_32_sum = sum(part_32_terms, np.zeros_like(A))
print("decomposition matches A:", np.allclose(part_32_sum, A))
print("rank(A):", np.linalg.matrix_rank(A))
print("number of terms:", len(part_32_terms))

# **Required LaTeX Expressions**

$$A\in\mathbb{R}^{3\times 4}$$

$$u^{(i)}v^{(i)\top}$$

$$\hat{e}=\frac{1}{\sqrt{a}}v$$

$$\operatorname{rank}(A)=3$$

$$\sum_{i=0}^{I-1}$$

---
# Section 5 --- Supervised or unsupervised? And the trap inside

*Round 1 asks which task is unsupervised. Grouping customers by purchasing behaviour with no predefined labels is the answer --- everything else in that list has a target column. But the interesting part is what happens when you actually run it.*

In [ ]:
rng = np.random.default_rng(5)
X_left = rng.normal(loc=(-2, -2), scale=0.2, size=(10, 2))
X_right = rng.normal(loc=(2, 2), scale=0.2, size=(10, 2))
X5 = np.vstack([X_left, X_right])
y5 = np.array([0] * 10 + [1] * 10)

class_centers = np.vstack([X5[y5 == k].mean(axis=0) for k in (0, 1)])
supervised_predictions = np.argmin(
    np.linalg.norm(X5[:, None, :] - class_centers[None, :, :], axis=2),
    axis=1,
)
print(np.mean(supervised_predictions == y5))

**Expected:** `1.0`. Two well-separated blobs, and a supervised model that saw the labels gets them all. Nothing surprising --- this is the control.

In [ ]:
# Initialize in reverse order so the arbitrary cluster IDs illustrate the label trap.
centers = np.vstack([X5[-1], X5[0]])
for _ in range(10):
    cluster_ids = np.argmin(
        np.linalg.norm(X5[:, None, :] - centers[None, :, :], axis=2),
        axis=1,
    )
    centers = np.vstack([X5[cluster_ids == k].mean(axis=0) for k in (0, 1)])

print("cluster ids:", cluster_ids[:10])
print("true labels:", y5[:10])

**Expected:** `cluster ids: [1 1 1 1 1 1 1 1 1 1]` against `true labels: [0 0 0 0 0 0 0 0 0 0]`. K-means never saw `y5`. It found the same two groups perfectly --- and named them backwards. It had no way to know which group deserves the name "0".

In [ ]:
raw_agreement = np.mean(cluster_ids == y5)
flipped_agreement = np.mean((1 - cluster_ids) == y5)
best_agreement = max(raw_agreement, flipped_agreement)
print(
    f"raw agreement {raw_agreement:.2f} | "
    f"flipped agreement {flipped_agreement:.2f} | "
    f"best {best_agreement:.2f}"
)

**Expected:** `raw agreement 0.00 | flipped agreement 1.00 | best 1.00`. **A cluster label is not a class label.** A perfect clustering can score 0% accuracy. When a problem asks you to cluster and then evaluate, you must match clusters to classes first --- by majority vote, or the Hungarian algorithm for more than two. Students lose real points to this every year.

---
# Section 6 --- L1 vs L2 at the same lambda

*Two models, same data, same $\lambda>0$: A uses L1, B uses L2. Which statement is most likely true? Rather than recite the answer, derive it in the one case where both have a closed form.*

In [ ]:
w_unregularized = np.array([-1.2, -0.3, 0.0, 0.4, 2.0])
lam = 0.5
w_ridge = w_unregularized / (1 + lam)
w_lasso = np.sign(w_unregularized) * np.maximum(
    np.abs(w_unregularized) - lam,
    0,
)
print("Ridge", np.round(w_ridge, 3))
print("Lasso", np.round(w_lasso, 3))

**Expected:** Ridge `[-0.8 -0.2  0.  0.267  1.333]`, Lasso `[-0.7 -0.  0.  0.  1.5]`. When the features are orthonormal, both penalties have exact solutions: L2 **divides** every weight by $1+\lambda$, while L1 **subtracts** $\lambda$ from its magnitude and stops at zero. Division never reaches zero; subtraction does.

In [ ]:
print("ridge:", np.count_nonzero(np.isclose(w_ridge, 0)))
print("lasso:", np.count_nonzero(np.isclose(w_lasso, 0)))

**Expected:** `ridge: 1 | lasso: 3`. Ridge's only zero was already zero before we started. Lasso created two more. This is the answer the problem wants: **at equal $\lambda$, L1 is the one that produces exactly-zero coefficients** --- it selects features, L2 only shrinks them.

In [ ]:
for current_lambda in (0.1, 0.5, 3.0):
    ridge_now = w_unregularized / (1 + current_lambda)
    lasso_now = np.sign(w_unregularized) * np.maximum(
        np.abs(w_unregularized) - current_lambda,
        0,
    )
    print(
        f"lambda={current_lambda:.1f} | "
        f"ridge zeros {np.count_nonzero(np.isclose(ridge_now, 0))} | "
        f"lasso zeros {np.count_nonzero(np.isclose(lasso_now, 0))}"
    )

**Expected:** lasso zeros go `1 -> 3 -> 5`; ridge stays at `1` throughout. The threshold is literally $\lambda$: any weight with $|w| \le \lambda$ is deleted. Ridge's count cannot move no matter how large $\lambda$ grows.

In [ ]:
import matplotlib.pyplot as plt

grid = np.linspace(-2, 2, 401)
ridge_map = grid / (1 + lam)
lasso_map = np.sign(grid) * np.maximum(np.abs(grid) - lam, 0)

plt.figure(figsize=(7, 4))
plt.plot(grid, ridge_map, label=r"$\ell_2$ shrinkage")
plt.plot(grid, lasso_map, label=r"$\ell_1$ soft thresholding")
plt.axhline(0, color="black", linewidth=0.7)
plt.axvline(0, color="black", linewidth=0.7)
plt.xlabel("Unregularized coefficient")
plt.ylabel("Regularized coefficient")
plt.title(r"Coefficient maps at $\lambda=0.5$")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

**Expected:** a plot where the L2 line is a straight line through the origin with a gentler slope, and the L1 line is **flat at zero** across $[-0.5, 0.5]$ before continuing at slope 1. That flat segment is the whole difference between the two methods, drawn.

---
# Section 7 --- Contest discipline

*These cost nothing to learn and cost points every year.*

In [ ]:
np.random.seed(42)
first_draw = np.random.random(3)
np.random.seed(42)
second_draw = np.random.random(3)
print(np.round(first_draw, 4))
print(np.round(second_draw, 4))
print("identical?", np.allclose(first_draw, second_draw))

**Expected:** `[0.3745 0.9507 0.732 ]` twice, `identical? True`. Graders re-run your notebook top to bottom. Without a seed, your reported number and their number differ, and you cannot argue about it. Seed once in the setup cell, always.

In [ ]:
restart_checks = [
    A.shape == (3, 4),
    np.allclose(A @ x, A_action(x)),
    np.linalg.matrix_rank(A) == 3,
    np.count_nonzero(np.isclose(w_lasso, 0)) == 3,
]
for check in restart_checks:
    print(check)

**Expected:** all four `True`. Before submitting: **Runtime -> Restart and run all**. A notebook that only works in the order you happened to click is a notebook that scores zero.

**Typeset, do not photograph.** Round 1 requires math in text cells, typeset. Handwriting an equation and pasting a photo is explicitly unacceptable. In a Colab text cell, `$A \in \mathbb{R}^{3\times 4}$` renders as $A \in \mathbb{R}^{3\times 4}$, and

```
$$A = \sum_{i=0}^{I-1} u^{(i)} v^{(i)\top}$$
```

renders as $$A = \sum_{i=0}^{I-1} u^{(i)} v^{(i)\top}$$

Also required: each part gets its own bold header cell, exactly like `# **Part 3.2**`. Graders search for these. A correct answer in an unlabelled cell is a correct answer they never find.

In [ ]:
weak = "I use three outer products, so the minimum value is three."
strong = (
    "Three outer products reproduce A, so I is at most three. Each outer "
    "product has rank at most one, so a sum of I such terms has rank at "
    "most I. Since A has three independent rows, its rank is three. "
    "Thus I is at least three, and both bounds prove the minimum is three."
)
print("weak ->", len(weak.split()), "words")
print("strong ->", len(strong.split()), "words")

**Expected:** `weak -> 11 words` and `strong -> 55 words`. Both give $I=3$. Only one earns full marks, because the question asked for the smallest $I$ and only the second shows nothing smaller exists. Achievability plus a lower bound --- that is the shape of every minimality argument you will write this year.

---
# Section 8 --- Your turn: a full solve

*A new mystery machine, start to finish. Recover the matrix, find the minimal decomposition, and justify minimality.*

In [ ]:
def B_action(t):
    t0, t1, t2 = t
    return np.array([2*t0 + t2, t0 - t1, 0, 3*t1], dtype=float)

basis3 = np.eye(3)
B = np.column_stack([B_action(e_j) for e_j in basis3])
print(B)
print("shape:", B.shape)

**Expected:**
```
[[ 2.  0.  1.]
 [ 1. -1.  0.]
 [ 0.  0.  0.]
 [ 0.  3.  0.]]
```
`shape: (4, 3)`. Same probing method: three basis vectors give three columns. Note the machine returns length 4 from length 3, so $B$ is $4\times 3$ --- outputs are rows.

In [ ]:
t = np.array([1., 2., 3.])
print(B @ t)
print(B_action(t))
print("match?", np.allclose(B @ t, B_action(t)))

**Expected:** both `[5. -1.  0.  6.]`, `match? True`. Always verify before decomposing --- a wrong $B$ makes every later part wrong too, and Part 2 usually carries more points than Part 1.

In [ ]:
rank_B = np.linalg.matrix_rank(B)
singular_values_B = np.linalg.svd(B, compute_uv=False)
nonzero_rows_B = np.count_nonzero(np.any(~np.isclose(B, 0), axis=1))
print("rank(B) =", rank_B)
print(np.round(singular_values_B, 3))
print("nonzero rows:", nonzero_rows_B)

**Expected:** `rank(B) = 3`, `[3.195 2.374 0.396]`, `nonzero rows: 3`. Row 2 is entirely zero, so the row-by-row construction needs only 3 terms --- and all three singular values are nonzero, so 3 is also the floor. Minimal $I = 3$.

Careful: $B$ has 4 rows but rank 3. Rank is capped by *both* dimensions, and here the 3 columns are the binding constraint.

In [ ]:
terms_B = []
for row_index, row in enumerate(B):
    if not np.allclose(row, 0):
        output_basis = np.eye(B.shape[0])[row_index]
        terms_B.append(np.outer(output_basis, row))

B_decomposed = sum(terms_B, np.zeros_like(B))
print("matches B?", np.allclose(B_decomposed, B))
print("terms used:", len(terms_B))

**Expected:** `matches B? True`, `terms used: 3`. Skipping the zero row is the small optimisation that turns a 4-term answer into a minimal 3-term one. Write the two-sided argument out in a text cell and you have a complete solution.

---
# **Your Turn A**

The input $t$ has three coordinates and $Bt$ has four coordinates, so $B\in\mathbb{R}^{4\times 3}$. Reading the coefficient of each input coordinate from the rule gives

$$
B=
\begin{pmatrix}
2&0&1\\
1&-1&0\\
0&0&0\\
0&3&0
\end{pmatrix}.
$$

For $t=(1,2,3)\top$,

$$
Bt=
\begin{pmatrix}
2(1)+3\\
1-2\\
0\\
3(2)
\end{pmatrix}
=
\begin{pmatrix}5\\-1\\0\\6\end{pmatrix},
$$

which agrees with the original rule.

For a minimal outer-product decomposition, let

$$
u^{(0)}=\begin{pmatrix}1\\0\\0\\0\end{pmatrix},\quad
v^{(0)}=\begin{pmatrix}2\\0\\1\end{pmatrix},\qquad
u^{(1)}=\begin{pmatrix}0\\1\\0\\0\end{pmatrix},\quad
v^{(1)}=\begin{pmatrix}1\\-1\\0\end{pmatrix},
$$

$$
u^{(2)}=\begin{pmatrix}0\\0\\0\\1\end{pmatrix},\qquad
v^{(2)}=\begin{pmatrix}0\\3\\0\end{pmatrix}.
$$

These three terms reproduce $B$, so $I\leq 3$. The $3\times 3$ submatrix formed by rows $1$, $2$, and $4$ has determinant $3\neq 0$, so $\operatorname{rank}(B)=3$. Since a sum of $I$ rank-one matrices has rank at most $I$, we also have $I\geq 3$. Therefore

$$\boxed{I=3}.$$

In [ ]:
t_check = np.array([1., 2., 3.])
print("B @ t:", B @ t_check)
print("rule:", B_action(t_check))
print("rank(B):", np.linalg.matrix_rank(B))
print("decomposition matches B:", np.allclose(B_decomposed, B))

# **Your Turn B**

Let $v=(1,-1,3)\top$ and $x=(2,5,-1)\top$. Since

$$\lVert v\rVert=\sqrt{1^2+(-1)^2+3^2}=\sqrt{11},$$

the unit vector is

$$
\hat e=\frac{1}{\sqrt{11}}\begin{pmatrix}1\\-1\\3\end{pmatrix}.
$$

Thus $a+b+c+d=11+1-1+3=\boxed{14}$.

The signed scalar projection is

$$
x\cdot\hat e
=\frac{2-5-3}{\sqrt{11}}
=\frac{-6}{\sqrt{11}},
$$

so $a+b=11-6=\boxed{5}$.

The residual after removing the component parallel to $\hat e$ is

$$
r=x-(x\cdot\hat e)\hat e
=x+\frac{6}{11}v
=\frac{1}{11}\begin{pmatrix}28\\49\\7\end{pmatrix}.
$$

Therefore $a+b+c+d=11+28+49+7=\boxed{95}$. Finally,

$$
r\cdot\hat e
=\frac{28-49+21}{11\sqrt{11}}
=0,
$$

so $r\perp\hat e$. This must be true because subtracting the full projection removes every component of $x$ in the $\hat e$ direction.

In [ ]:
v = np.array([1., -1., 3.])
x_projection = np.array([2., 5., -1.])
e_hat = v / np.linalg.norm(v)
scalar_projection = x_projection @ e_hat
residual = x_projection - scalar_projection * e_hat
print("e_hat:", e_hat)
print("scalar projection:", scalar_projection)
print("residual:", residual)
print("residual dot e_hat:", residual @ e_hat)

# **Your Turn C**

The answer is **B: training an autoencoder to reconstruct its own input images**.

An autoencoder does have something it tries to match, but that target is generated from the input itself rather than supplied as a separate human-labeled class or value. It learns the structure of unlabeled data, so it is treated as unsupervised or self-supervised learning in the usual sense.

In [ ]:
options = {
    "A": "temperature prediction from paired historical readings",
    "B": "autoencoder reconstruction from unlabeled images",
    "C": "pixel classification from object labels",
    "D": "essay scoring from teacher scores",
    "E": "drug-response prediction from labeled outcomes",
}
answer = "B"
print(answer, "-", options[answer])

# **Your Turn D**

Under an orthonormal design, the $\ell_1$ solution applies soft thresholding:

$$
\widehat w_j^{(\ell_1)}
=\operatorname{sign}(w_j)\max\{|w_j|-\lambda,0\}.
$$

The coefficient magnitudes are

$$0.15,\ 0.4,\ 0.8,\ 1.9,\ 2.5.$$

At $\lambda=1.0$, exactly the coefficients with magnitudes $0.15$, $0.4$, and $0.8$ become zero. Therefore the answer to part (a) is **C**, and exactly three coefficients are zero whenever

$$
\boxed{0.8\leq\lambda<1.9}.
$$

For $\ell_2$ regularization under the same orthonormal design,

$$
\widehat w_j^{(\ell_2)}=\frac{w_j}{1+\lambda}.
$$

None of the original coefficients is zero, and division by a finite positive number cannot make a nonzero value exactly zero. Therefore $\ell_2$ gives zero exact zeros at each listed value $\lambda\in\{0.1,0.5,1.0,2.0,3.0\}$.

In [ ]:
w_turn_d = np.array([-2.5, 0.8, -0.4, 1.9, 0.15])
listed_lambdas = [0.1, 0.5, 1.0, 2.0, 3.0]

for current_lambda in listed_lambdas:
    l1_solution = np.sign(w_turn_d) * np.maximum(
        np.abs(w_turn_d) - current_lambda,
        0,
    )
    l2_solution = w_turn_d / (1 + current_lambda)
    print(
        f"lambda={current_lambda:.1f}: "
        f"l1 zeros={np.count_nonzero(np.isclose(l1_solution, 0))}, "
        f"l2 zeros={np.count_nonzero(np.isclose(l2_solution, 0))}"
    )

---
## Submission check

- Core notebook completed through Section 8
- Part 3.2 includes both achievability and minimality
- All five required expressions are typeset in LaTeX
- Your Turns A, B, C, and D are typed and supported by code